In [ ]:
import sys
from pathlib import Path

ROOT = Path("/home/jovyan/work/")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.spark import get_spark
from src.io import read_parquet, write_parquet, to_csv_via_pandas
import src.config as cfg
from src.utils import print_header, shape 

from src.metrics import (
    calculate_credit_ratios_and_utilization,
    calculate_risk,
    calculate_debt_and_net_income,
    apply_cap_and_binning, 
    calculate_caq_segment,
    calculate_grouped_compliance_rates
)

from src.engagement import (
    calculate_full_engagement_analysis,
    print_engagement_report,
    join_product_info
)


In [ ]:
spark = get_spark("MetricsAndEngagementPipeline")

# Leer DataFrames
df_features = read_parquet(spark, cfg.COMBINED_PATH)
df_clients = read_parquet(spark, cfg.CLIENTS_PATH_CLEAN)

print_header("Carga de Datos")
print(f"Features Base: {df_features.count():,} filas")


Carga de Datos
Features Base: 1,688,934 filas


In [3]:
print_header("1. Feature Engineering Financiero")

# 1. Ratios, Riesgo, Deuda
df_features = calculate_credit_ratios_and_utilization(df_features)
df_features = calculate_debt_and_net_income(df_features)
df_features = calculate_risk(df_features) # RM Score básico

# 2. Transformaciones y CAQ
df_features = apply_cap_and_binning(df_features, num_prods_cap=10)
df_features = calculate_caq_segment(df_features, ["TOTAL_INCOME", "CREDIT_CARD_LIMIT", "PAYMENT_RATIO"])

# Muestra parcial
df_features.select("CLIENT_ID", "PAYMENT_RATIO", "CUR", "RM_SCORE", "CAQ_SEGMENT").show(3)


1. Feature Engineering Financiero
+------------+-------------+-------------------+--------+-----------+
|   CLIENT_ID|PAYMENT_RATIO|                CUR|RM_SCORE|CAQ_SEGMENT|
+------------+-------------+-------------------+--------+-----------+
|ES182100115E|          0.0|0.34496913580246913|       1| CAQ_Normal|
|ES182100180D|          1.0|                0.0|       0|   CAQ_Alto|
|ES182100180D|          1.0|                0.0|       0|   CAQ_Alto|
+------------+-------------+-------------------+--------+-----------+
only showing top 3 rows



In [4]:
# Calcula el DataFrame específico de Engagement con sus 3 dimensiones y scores
df_engagement_score = calculate_full_engagement_analysis(df_features)

# Imprime el reporte detallado (texto)
print_engagement_report(df_engagement_score)

ANÁLISIS DE ENGAGEMENT (3 DIMENSIONES)

1. VERIFICACIÓN DE DATOS DISPONIBLES (ENGAGEMENT)
------------------------------------------------------------
 Todas las columnas necesarias están disponibles
 Total registros: 1,688,934

--- REPORTE DE RESULTADOS (45,668 clientes) ---

1. DISTRIBUCIÓN ENGAGEMENT TOTAL:
   • ENGAGEMENT ALTO: 9,302 (20.4%)
   • ENGAGEMENT MEDIO-ALTO: 10,215 (22.4%)
   • ENGAGEMENT MEDIO: 8,779 (19.2%)
   • ENGAGEMENT BAJO: 3,523 (7.7%)
   • ENGAGEMENT MUY BAJO: 13,849 (30.3%)

2. PROMEDIOS: Total 52.9 | Var 94.4 | Int 35.0 | Hab 35.3

3. TOP 3 CLIENTES (ENGAGEMENT TOTAL):
+------------+----------------------+-----+
|CLIENT_ID   |NIVEL_ENGAGEMENT_TOTAL|SCORE|
+------------+----------------------+-----+
|ES182116388P|ENGAGEMENT ALTO       |258.5|
|ES182283133M|ENGAGEMENT ALTO       |234.0|
|ES182424510R|ENGAGEMENT ALTO       |231.0|
+------------+----------------------+-----+



In [5]:
print_header("2. CONSOLIDACIÓN FINAL Y GUARDADO")

# 1. UNIR EL SCORE DE ENGAGEMENT AL DF PRINCIPAL
# Usamos un LEFT JOIN para enriquecer df_features con las nuevas columnas de engagement
# sin perder ninguna métrica calculada en el paso 3.
df_final = df_features.join(
    df_engagement_score.select(
        "CLIENT_ID", 
        "ENGAGEMENT_SCORE_TOTAL", "NIVEL_ENGAGEMENT_TOTAL",
        "SCORE_VARIEDAD", "NIVEL_VARIEDAD",
        "SCORE_INTENSIDAD", "NIVEL_INTENSIDAD",
        "SCORE_HABITOS_SALUDABLES", "NIVEL_HABITOS"
    ),
    on="CLIENT_ID",
    how="left"
)

# 2. Agregar información de Producto (para análisis posteriores)
df_final_completo = join_product_info(df_final, df_clients)

df_final_completo = df_final_completo\
    .drop("GENDER", "AGE_IN_YEARS", "FAMILY_SIZE", "EDUCATION", "MARITAL_STATUS", "HOME_SENIORITY",
          "HOME_SITUATION", "REGION_SCORE", "OCCUPATION")

# 3. Guardar el resultado maestro
write_parquet(df_final_completo, cfg.FINAL_FEATURES_PATH, mode="overwrite")

print(f"✅ Pipeline completado.")
print(f"Features Finales guardados en: {cfg.FINAL_FEATURES_PATH}")
print(f"Dimensiones finales: {shape(df_final_completo)}")

# Muestra final para verificar
df_final_completo.select(
    "CLIENT_ID", "CAQ_SEGMENT", "RM_SCORE", "ENGAGEMENT_SCORE_TOTAL", "PRODUCT_TYPE"
).show(5, truncate=False)


2. CONSOLIDACIÓN FINAL Y GUARDADO
✅ Pipeline completado.
Features Finales guardados en: /home/jovyan/work/data/FINAL_FEATURES
Dimensiones finales: (1688934, 65)
+------------+-----------+--------+----------------------+------------+
|CLIENT_ID   |CAQ_SEGMENT|RM_SCORE|ENGAGEMENT_SCORE_TOTAL|PRODUCT_TYPE|
+------------+-----------+--------+----------------------+------------+
|ES182100115E|CAQ_Normal |1       |50.5                  |PRODUCT 1   |
|ES182100180D|CAQ_Alto   |0       |16.5                  |PRODUCT 1   |
|ES182100180D|CAQ_Alto   |0       |16.5                  |PRODUCT 1   |
|ES182100180D|CAQ_Normal |0       |16.5                  |PRODUCT 1   |
|ES182100180D|CAQ_Alto   |0       |16.5                  |PRODUCT 1   |
+------------+-----------+--------+----------------------+------------+
only showing top 5 rows



In [ ]:
final_feature = read_parquet(spark, cfg.FINAL_FEATURES_PATH)

to_csv_via_pandas(final_feature, cfg.FINAL_FEATURES_CSV, index=False)